# Step 6 — General Instructions (Rules, Definitions, Defaults, Tone)

**Mistplay Genie Workshop**

After Steps 3-5, what remains are business rules that **aren't** math, a filter, a join, or a single example. These need natural-language **instructions**:

- **Fiscal calendar** — fiscal year starts Feb 1 (Q1 = Feb-Apr …).
- **Default exclusion** — exclude internal/test accounts (`is_internal = true`) unless asked.
- **Definitions** — what "active customer" and "revenue" mean.
- **Tone** — concise, lead with the number, state applied filters.

This is the point of the whole progression: **instructions are the residual layer**. Reach for them only when a structured asset (value matching, metric view, example) can't express the rule — over-using instructions is the #1 cause of inconsistent Genie behavior.

> **API vs UI:** general instructions map to `instructions.text_instructions` — set via REST `set_instructions` + PATCH. All in code.

In [0]:
import os, sys, importlib
nb_dir = os.getcwd()
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import workshop_lib.config, workshop_lib.genie_builder, workshop_lib.instructions, workshop_lib.questions
for m in (workshop_lib.config, workshop_lib.genie_builder, workshop_lib.instructions, workshop_lib.questions):
    importlib.reload(m)
from workshop_lib.config import WorkshopConfig
from workshop_lib import genie_builder as gb
from workshop_lib import instructions as instr
from workshop_lib import questions
# Load config saved by notebook 01 — no widgets needed.
cfg = WorkshopConfig.from_saved(spark)
client = gb.GenieClient()
space_id = gb.require_space_id(spark, cfg)
print("space:", space_id)

## 1. Review the instruction text

Open `workshop_lib/instructions.py` — this is the exact guidance we'll set. Keep it tight and rule-focused.

In [0]:
print(instr.INSTRUCTIONS)

## 2. Set instructions and patch

In [0]:
builder = client.load_builder(space_id)
instr.add_instructions(builder)
print(builder.summary())
client.patch_space(space_id, title=cfg.genie_space_name,
                   description="Mistplay Genie Workshop — production-grade (instructions added).",
                   warehouse_id=cfg.warehouse_id, serialized_space=builder.to_serialized())
print("Patched.")

## 3. Verify

In [0]:
d = client.load_builder(space_id).to_dict()
ti = d.get("instructions",{}).get("text_instructions",[])
print("text_instructions:", len(ti))
print((ti[0]["content"][0][:200] + "...") if ti else "(none)")

## 4. Try it — the two question sets

In [0]:
# Questions with trap answers. Set A should work; Set B are traps
# (agent often returns the 'naive_or_trap_answer').
print(questions.render(6))
print('\n--- Trap answers from YOUR data ---')
display(questions.answer_df(spark, cfg, 6).select("set", "question", "naive_or_trap_answer", "why_agent_fails"))

## Workshop complete 🎉

Your Genie space went from a bare table list to a production-grade, **benchmarked** space:

1. Documented gold data → 2. bare space → 3. value matching → 4. metric views (joins + filters) → 5. curated examples → 6. instructions → 7. benchmarks.

**Next (advanced part):** open the space in **Genie Workbench**, run an **IQ scan** for an objective score, and use **auto-optimize** against this benchmark set to close the remaining gaps.
**alternatively:** open the space and use genie code to improve with Genie code.